# Longevity Analysis with LIFE-M Demo Data 

**Author:** Julie Huang  
**Date:** March 2026  

--- 

## Installation note

This notebook demonstrates usage of the `postlink` package.

Because `postlink` depends on `rstan`, it must be installed in a
local R environment (e.g., RStudio). Installation inside hosted
Jupyter environments is not currently supported.

```r
install.packages("remotes")
remotes::install_github("postlink-group/postlink")

library(postlink)

In [ ]:
data("lifem", package = "postlink")

# Overview

This example illustrates how to estimate the relationship between age at death and year of birth using data from the LIFE-M Project. The example compares three approaches:

1. Naive Linear Regression (no correction for linkage error)

2. Hand-Linked Only Regression (restricting to high-quality manually linked records)

3. Adjustment Approach (modeling and adjusting for probabilistic linkage errors using postlink)
<br>

# 1. DataSet: `LIFE-M` Project

### Dataset Overview

The lifem dataset originates from the LIFE-M Project, which links historical birth and death certificates for individuals born in Ohio (1883–1906). It contains 3,238 matched records, each representing a unique individual. The dataset was constructed through both manual (“hand-linked”) and automated (“machine-linked”) record linkage methods.

A subset of 2,159 records were hand-linked at some level (HL) — meaning they were reviewed and confirmed by human researchers — while the remaining 1,079 records were machine-linked (ML) based on probabilistic matching without clerical review. The LIFE-M team estimates that the overall mismatch rate is approximately 5% (Bailey et al., 2022).

This dataset was curated for methodological research in record linkage error adjustment, as detailed in Slawski et al., 2024, and is distributed with the `postlink` R package for demonstration purposes.

### Variables

`yob` (int): Year of birth, ranging from 1883 to 1906  
`unit_yob` (db): Year of birth rescaled to the [0, 1] interval for numerical stability in regression  
`age_at_death` (int): Age at death in years  
`hndlnk` (char): Linkage type indicator — `Hand-Linked At Some Level` if manually verified, otherwise `machine-linked`  
`commf` (db): “Commonness” score of the first name — computed as the ratio of log-counts of the name in the 1940 Census to the most frequent name in that Census  
`comml` (db): “Commonness” score of the last name, defined analogously to `commf`  

### Sample Records

In [ ]:
head(lifem, 5)

<br>

# 2. Linear Regression: Age at Death vs. Year of Birth

### Objective
This section investigates the relationship between individuals’ year of birth and their age at death using data from the LIFE–M Project. Three regression approaches are compared to evaluate how record linkage quality and probabilistic adjustment affect model estimates and inference robustness.
<br><br>

### Model Specification
We model the relationship between year of birth and age at death while accounting for potential linkage errors.  
Let $y_i$ denote the observed age at death and $x_i$ the rescaled year of birth (`unit_yob`).  

The true data-generating process is specified as follows:

1. $y_i \mid m_i = 0, x_i \sim N(\beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3, \sigma^2)$  
2. $y_i \mid m_i = 1, x_i \sim N(\mu, \tau^2)$  
3. $m_i \mid \text{commf}, \text{comml} \sim \text{Bernoulli}\left( \dfrac{\exp(\gamma_0 + \gamma_1 \text{commf} + \gamma_2 \text{comml})}{1 + \exp(\gamma_0 + \gamma_1 \text{commf} + \gamma_2 \text{comml})} \right)$  
4. Anticipated overall mismatch rate ≈ 5%  
5. Hand-linked records are assumed to be correctly matched ($m_i = 0$)
<br><br>

### 2.1 Naive Approach

#### Methodology
The naive approach fits a cubic polynomial regression model using all available records—both hand-linked and machine-linked—without any adjustment for potential linkage errors:

$$
\text{age\_at\_death}_i = \beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3 + \varepsilon_i, \quad \varepsilon_i \sim N(0, \sigma^2)
$$

where $x_i$ denotes the rescaled year of birth (`unit_yob`)

#### R Implementation

In [ ]:
fit <- lm(age_at_death ~ poly(unit_yob, 3, raw = TRUE), data = lifem)
summary(fit)

#### Results
- R2 = 0.0936, indicating limited explanatory power
- All polynomial terms (linear, quadratic, cubic) are statistically significant (p < 0.001), suggesting a nonlinear relationship between year of birth and age at death
- Residual standard error: 19.67 years (df = 3234)

#### Interpretation
The model suggests that later birth cohorts (higher `unit_yob`) tend to live longer, but the curvature terms imply the trend is not purely linear. However, since linkage errors are unaccounted for, these coefficients may be biased downward or distorted, particularly if mislinked records introduce noise into the outcome variable.
<br><br>

### 2.2 Hand-Linked Only Approach

#### Methodology
The hand-linked only approach restricts the analysis to records manually verified by human researchers (“Hand-Linked At Some Level”). By removing machine-linked cases, the model minimizes the risk of mismatched pairs, assuming all hand-linked records are accurate.

#### R Implementation

In [ ]:
fit <- lm(age_at_death ~ poly(unit_yob, 3, raw = TRUE),
     data = lifem[lifem$hndlnk == "Hand-Linked At Some Level",])
summary(fit)

#### Results
- R2 = 0.119, slightly higher than the naive approach, reflecting a modest improvement in explanatory power
- All polynomial coefficients remain significant (p < 0.01)
- Residual standard error: 18.98 (df = 2155)

#### Interpretation
The model fitted on hand-linked data yields stronger effects and a better fit, suggesting that excluding uncertain matches reduces noise. However, this restriction sacrifices sample size and may introduce selection bias if manually linked cases are not representative of the entire dataset (e.g., overrepresentation of common names).
<br><br>

### 2.3 Adjustment Approach

#### Methodology
The adjustment approach uses a two-phase workflow in postlink: first we specify a linkage-error adjustment object via `adjMixture()`, then we fit the downstream model with `plglm()` using that adjustment. This method models observed outcomes as a mixture of correctly linked and mismatched records:
$$
y_i =
\begin{cases}
N(\beta_0 + \beta_1 x_i + \beta_2 x_i^2 + \beta_3 x_i^3, \sigma^2), & \text{if correctly linked } (m_i = 0) \\
N(\mu, \tau^2), & \text{if mismatched } (m_i = 1)
\end{cases}
$$
where the mismatch indicator $m_i$ is modeled as a Bernoulli variable with logit link, using name commonness covariates (`commf`, `comml`) specified by `mformula`.
An anticipated mismatch rate of 5% is assumed, consistent with LIFE–M project documentation.

#### R Implementation

In [ ]:
adj <- adjMixture(
  data = lifem,
  mformula = ~ commf + comml, # mismatch model
  mrate    = 0.05, # assumed mismatch rate
  safematches = (lifem$hndlnk == "Hand-Linked At Some Level") # “safe matches” (hand-linked treated as correct)
)

print(adj) # inspect the adjustment specification before fitting

# Fit adjusted model (postlink)
fit <- plglm(
  formula    = age_at_death ~ poly(unit_yob, 3, raw = TRUE),
  data       = lifem,
  adjustment = adj,
  family     = gaussian()
)

# Post-fit outputs
summary(fit)
confint(fit)

#### Results
- Adjusted estimates are similar in direction but smaller in magnitude than the unadjusted ones, reflecting attenuation due to mismatch correction
- The average estimated correct match rate is 0.829, indicating roughly 83% of records are correctly linked after probabilistic adjustment
- Linkage covariates (`commf`, `comml`) have positive coefficients, suggesting that less common names correspond to a higher probability of correct matching

#### Interpretation
By incorporating linkage uncertainty, the mixture model provides more realistic inference that balances bias (from mislinks) and variance (from reduced sample size). The approach corrects for measurement error introduced during record linkage, leading to estimates more representative of the underlying population-level relationship.